# PH2_NB6b — Track A Diagnosis and Fix

Track A's first run landed in the pre-registered third scenario: the hybrid scored **below** the
probe on the standard split (97.5 vs 99.7), which the work plan says means *investigate scaling /
alignment / training before concluding anything about fusion*. This notebook is that investigation.

## Why the first run is not evidence against fusion

Two variables changed at once between the probe and the Track-A hybrid: the five statistical
features were added, **and** the classifier changed from a converged logistic regression to an MLP
head. The MLP was trained full-batch — one optimizer step per epoch — and early-stopped around
epoch ~28, so the head received roughly **thirty gradient steps** against LogReg's hundreds of
converged LBFGS iterations, with val-F1 still climbing at the stop. Under-training, dropout noise on
a near-linearly-separable task, and LayerNorm's re-scaling of the concatenated geometry are all
plausible culprits — none of which is the fusion idea itself.

## The diagnostic ladder (single-variable steps)

| # | Classifier | Input | Question it answers |
|---|-----------|-------|---------------------|
| 1 | LogReg | 768 (neural) | reproduce the probe (sanity: expect ~99.7) |
| 2 | LogReg | 773 (hybrid) | do the 5 features hurt *by themselves*? |
| 3 | fixed MLP | 768 (neural) | is the head the problem? |
| 4 | fixed MLP | 773 (hybrid) | the repaired Track-A hybrid |

Plus: **LOGO for ladder step 2** (cheap). If LogReg-on-773 lifts the worst fold above the probe's
92.7, the central fusion hypothesis works and the first run's failure was entirely the head.

The **fixed trainer** uses shuffled mini-batches (256) — thousands of updates instead of thirty —
up to 200 epochs, patience 20, and sweeps dropout {0.0, 0.3} and LayerNorm {on, off} so every design
guess is measured, not assumed.

**Inputs:** same three datasets as PH2_NB6. **Outputs:** `ph2_nb6b_ladder.parquet`,
`ph2_nb6b_logo.parquet`, `ph2_nb6b_best_head.pt`.

## Setup and alignment (identical contract to PH2_NB6)

In [1]:
import pandas as pd, numpy as np, os, glob, time, json
import torch, torch.nn as nn

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = '/kaggle/working'
print('device:', DEVICE)

def find_file(preferred, pattern, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find_file('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', '*.parquet', 'dataset')
VSTAT = find_file('/kaggle/input/notebooks/bahaaqassem/nb4-extract-vstat/vstat_scaled.parquet', '*.parquet', 'vstat_scaled', 'vstat')
EMB   = find_file('/kaggle/input/notebooks/bahaaqassem/nb5c-electra-camelbert/nb5c_camelbert_msa_cls_frozen.npy', '*.npy',
                  'camelbert_msa_cls', 'camelbert')

df  = pd.read_parquet(DATA)
vs  = pd.read_parquet(VSTAT)
emb = np.load(EMB).astype(np.float32)
FEATURES = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']

assert len(df) == len(emb)
vs_aligned = vs.set_index('article_id').loc[df['article_id']].reset_index()
assert (vs_aligned['article_id'].to_numpy() == df['article_id'].to_numpy()).all()
assert (vs_aligned['label'].to_numpy() == df['label'].to_numpy()).all()
Xstat = vs_aligned[FEATURES].to_numpy(dtype=np.float32)
Xhyb  = np.concatenate([emb, Xstat], axis=1)
y     = df['label'].to_numpy()
splits = df['split'].to_numpy(); gens = df['generator'].to_numpy()
tr_m, va_m, te_m = splits=='train', splits=='val', splits=='test'
print('ALIGNMENT OK |', Xhyb.shape)

device: cpu
ALIGNMENT OK | (7101, 773)


## Fixed MLP trainer — mini-batches, proper convergence

Same architecture family as the plan (optional LayerNorm -> Linear(d->H) -> ReLU -> optional
Dropout -> Linear(H->2)), but trained the way a small head should be: shuffled mini-batches of 256
(~21 steps per epoch), AdamW 1e-3, up to 200 epochs, early stopping on val macro-F1 with patience
20, best weights restored.

In [2]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression

class Head(nn.Module):
    def __init__(self, d, hidden=256, dropout=0.3, layernorm=True):
        super().__init__()
        layers = []
        if layernorm: layers.append(nn.LayerNorm(d))
        layers += [nn.Linear(d, hidden), nn.ReLU()]
        if dropout > 0: layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def class_weights(ytr):
    n0, n1 = int((ytr==0).sum()), int((ytr==1).sum())
    return torch.tensor([(n0+n1)/(2*n0), (n0+n1)/(2*n1)], dtype=torch.float, device=DEVICE)

def train_mlp(Xtr, ytr, Xva, yva, hidden=256, dropout=0.3, layernorm=True,
              batch=256, max_epochs=200, patience=20, seed=SEED):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = Head(Xtr.shape[1], hidden, dropout, layernorm).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    lossf = nn.CrossEntropyLoss(weight=class_weights(ytr))
    Xtr_t = torch.tensor(Xtr, device=DEVICE); ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Xva_t = torch.tensor(Xva, device=DEVICE)
    n = len(Xtr_t)
    best_f1, best_state, best_ep, wait = -1.0, None, -1, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(n, generator=g).to(DEVICE)
        for b in range(0, n, batch):
            idx = perm[b:b+batch]
            opt.zero_grad()
            loss = lossf(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pva = model(Xva_t).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pva, average='macro')
        if f1 > best_f1 + 1e-6:
            best_f1, best_ep, wait = f1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, best_f1, best_ep

def score_mlp(model, X, yy):
    model.eval()
    with torch.no_grad():
        proba = torch.softmax(model(torch.tensor(X, device=DEVICE)), 1)[:, 1].cpu().numpy()
    pred = (proba >= 0.5).astype(int)
    return {'accuracy': accuracy_score(yy, pred), 'precision': precision_score(yy, pred),
            'recall': recall_score(yy, pred), 'macro_f1': f1_score(yy, pred, average='macro'),
            'auc_roc': roc_auc_score(yy, proba)}, pred

def score_lr(Xtr, ytr, Xte, yte_):
    m = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED).fit(Xtr, ytr)
    proba = m.predict_proba(Xte)[:, 1]; pred = (proba >= 0.5).astype(int)
    return {'accuracy': accuracy_score(yte_, pred), 'precision': precision_score(yte_, pred),
            'recall': recall_score(yte_, pred), 'macro_f1': f1_score(yte_, pred, average='macro'),
            'auc_roc': roc_auc_score(yte_, proba)}, m

print('fixed trainer + LogReg scorer ready')

fixed trainer + LogReg scorer ready


## The ladder — steps 1 and 2 (LogReg on 768 vs 773)

Step 1 must reproduce the probe; step 2 changes exactly one thing (the five extra dimensions). If
step 2 holds ~99.7, the features are innocent and everything lost in the first run was the head.

In [3]:
Xn = emb   # 768 neural-only
ladder = []

r1, _ = score_lr(Xn[tr_m], y[tr_m], Xn[te_m], y[te_m])
ladder.append({'step': '1. LogReg / neural 768', **r1})
print(f"1. LogReg/768   macroF1 {100*r1['macro_f1']:.1f}%   (probe reference: 99.7)")

r2, lr_hyb = score_lr(Xhyb[tr_m], y[tr_m], Xhyb[te_m], y[te_m])
ladder.append({'step': '2. LogReg / hybrid 773', **r2})
d21 = 100*(r2['macro_f1'] - r1['macro_f1'])
print(f"2. LogReg/773   macroF1 {100*r2['macro_f1']:.1f}%   (delta vs step1: {d21:+.1f})")
if d21 >= -0.3:
    print('   -> the five features do NOT hurt a converged linear classifier')
else:
    print('   -> the features cost a converged linear head; scaling interaction suspected')

1. LogReg/768   macroF1 99.7%   (probe reference: 99.7)
2. LogReg/773   macroF1 99.6%   (delta vs step1: -0.1)
   -> the five features do NOT hurt a converged linear classifier


## The ladder — steps 3 and 4 (fixed MLP on 768 vs 773)

With the trainer repaired, the head configuration itself is swept: dropout {0.0, 0.3} x layernorm
{on, off}, selected on validation, separately for each input. This answers both "was the first run
under-trained?" and "which head design should Track A/B use?".

In [4]:
def best_mlp_for(X, label):
    rows = []
    for do in [0.0, 0.3]:
        for ln in [True, False]:
            t0 = time.time()
            m, vf1, ep = train_mlp(X[tr_m], y[tr_m], X[va_m], y[va_m],
                                   hidden=256, dropout=do, layernorm=ln)
            rows.append({'input': label, 'dropout': do, 'layernorm': ln,
                         'val_f1': vf1, 'epoch': ep, 'sec': round(time.time()-t0,1), 'model': m})
            print(f'  {label} do={do} ln={ln}: val_f1 {100*vf1:.2f}% (ep {ep}, {rows[-1]["sec"]}s)')
    best = max(rows, key=lambda r: r['val_f1'])
    print(f'  -> best {label}: dropout={best["dropout"]} layernorm={best["layernorm"]}')
    return best, rows

print('MLP config sweep — neural 768:')
best_n, rows_n = best_mlp_for(Xn, 'neural768')
print('MLP config sweep — hybrid 773:')
best_h, rows_h = best_mlp_for(Xhyb, 'hybrid773')

r3, _ = score_mlp(best_n['model'], Xn[te_m], y[te_m])
ladder.append({'step': f'3. fixed MLP / neural 768 (do={best_n["dropout"]}, ln={best_n["layernorm"]})', **r3})
r4, pred4 = score_mlp(best_h['model'], Xhyb[te_m], y[te_m])
ladder.append({'step': f'4. fixed MLP / hybrid 773 (do={best_h["dropout"]}, ln={best_h["layernorm"]})', **r4})

print(f"\n3. fixedMLP/768 macroF1 {100*r3['macro_f1']:.1f}%")
print(f"4. fixedMLP/773 macroF1 {100*r4['macro_f1']:.1f}%   (first run was 97.5)")

lad = pd.DataFrame([{k:v for k,v in row.items() if k!='model'} for row in ladder])
show = lad.copy()
for c in ['accuracy','precision','recall','macro_f1','auc_roc']:
    show[c] = (100*show[c]).round(1)
print('\n' + show[['step','macro_f1','auc_roc','accuracy']].to_string(index=False))
lad.to_parquet(f'{OUT_DIR}/ph2_nb6b_ladder.parquet', index=False)

cfg = {'hidden':256, 'dropout':best_h['dropout'], 'layernorm':best_h['layernorm']}
torch.save({'state_dict': best_h['model'].state_dict(), **cfg}, f'{OUT_DIR}/ph2_nb6b_best_head.pt')
print('\nselected hybrid head config for Track A/B:', cfg)

MLP config sweep — neural 768:
  neural768 do=0.0 ln=True: val_f1 99.69% (ep 24, 12.2s)
  neural768 do=0.0 ln=False: val_f1 99.69% (ep 20, 4.1s)
  neural768 do=0.3 ln=True: val_f1 99.69% (ep 18, 6.2s)
  neural768 do=0.3 ln=False: val_f1 99.53% (ep 6, 3.5s)
  -> best neural768: dropout=0.0 layernorm=True
MLP config sweep — hybrid 773:
  hybrid773 do=0.0 ln=True: val_f1 99.69% (ep 37, 7.7s)
  hybrid773 do=0.0 ln=False: val_f1 99.38% (ep 6, 2.6s)
  hybrid773 do=0.3 ln=True: val_f1 99.69% (ep 19, 6.7s)
  hybrid773 do=0.3 ln=False: val_f1 99.53% (ep 15, 4.9s)
  -> best hybrid773: dropout=0.0 layernorm=True

3. fixedMLP/768 macroF1 99.8%
4. fixedMLP/773 macroF1 99.7%   (first run was 97.5)

                                       step  macro_f1  auc_roc  accuracy
                     1. LogReg / neural 768      99.7    100.0      99.7
                     2. LogReg / hybrid 773      99.6    100.0      99.6
3. fixed MLP / neural 768 (do=0.0, ln=True)      99.8    100.0      99.8
4. fixed MLP /

## LOGO, twice: LogReg-hybrid (step 2) and fixed-MLP-hybrid (step 4)

The cheap LogReg-hybrid LOGO tests the fusion hypothesis with a fully-converged classifier — if the
worst fold beats the probe's 92.7, fusion helps robustness, full stop. The fixed-MLP LOGO then shows
whether the chosen head preserves that gain.

In [5]:
camel_probe_ref = {'deepseek': 98.3, 'gemini': 99.3, 'gpt': 92.7, 'opus': 99.4,
                   'qwen': 98.3, 'sonnet': 99.6}   # verify against the NB5c log for the thesis table

gen_list = sorted(df.loc[df['label']==1, 'generator'].unique().tolist())

def logo_masks(g):
    trm = (splits=='train') & ((y==0) | (gens!=g))
    vam = (splits=='val')   & ((y==0) | (gens!=g))
    tem = (splits=='test')  & ((y==0) | (gens==g))
    return trm, vam, tem

rows = []
for g in gen_list:
    trm, vam, tem = logo_masks(g)
    # arm 1: converged LogReg on hybrid
    rA, _ = score_lr(Xhyb[trm], y[trm], Xhyb[tem], y[tem])
    # arm 2: fixed MLP on hybrid (selected config)
    mB, _, _ = train_mlp(Xhyb[trm], y[trm], Xhyb[vam], y[vam], hidden=256,
                         dropout=best_h['dropout'], layernorm=best_h['layernorm'])
    rB, pB = score_mlp(mB, Xhyb[tem], y[tem])
    cmg = confusion_matrix(y[tem], pB, labels=[0,1])
    rows.append({'held_out': g, 'probe_ref': camel_probe_ref.get(g, np.nan),
                 'logreg_hybrid_f1': rA['macro_f1'], 'mlp_hybrid_f1': rB['macro_f1'],
                 'mlp_ai_recall': cmg[1,1]/max(cmg[1].sum(),1),
                 'mlp_human_recall': cmg[0,0]/max(cmg[0].sum(),1)})
    print(f"{g:<10} probe {camel_probe_ref.get(g,'?'):>5}%   "
          f"LogReg-hyb {100*rA['macro_f1']:5.1f}%   MLP-hyb {100*rB['macro_f1']:5.1f}%")

logo = pd.DataFrame(rows)
print('\nsummary (macro-F1):')
print(f"  probe (reference):  mean 98.1  worst 92.7 (gpt)")
print(f"  LogReg-hybrid:      mean {100*logo['logreg_hybrid_f1'].mean():.1f}  "
      f"worst {100*logo['logreg_hybrid_f1'].min():.1f} "
      f"({logo.loc[logo['logreg_hybrid_f1'].idxmin(),'held_out']})")
print(f"  fixed-MLP-hybrid:   mean {100*logo['mlp_hybrid_f1'].mean():.1f}  "
      f"worst {100*logo['mlp_hybrid_f1'].min():.1f} "
      f"({logo.loc[logo['mlp_hybrid_f1'].idxmin(),'held_out']})")

wl = logo['logreg_hybrid_f1'].min()
if wl > 0.927 + 0.002:
    print('\nFUSION HYPOTHESIS SUPPORTED: converged hybrid beats the probe on the worst fold')
elif wl >= 0.927 - 0.002:
    print('\nWorst-fold PARITY with the probe under a converged classifier')
else:
    print('\nEven a converged hybrid does not lift the worst fold — fusion gain not shown on LOGO')

logo.to_parquet(f'{OUT_DIR}/ph2_nb6b_logo.parquet', index=False)
print('\nsaved ph2_nb6b_ladder.parquet, ph2_nb6b_logo.parquet, ph2_nb6b_best_head.pt')

deepseek   probe  98.3%   LogReg-hyb  96.9%   MLP-hyb  95.6%
gemini     probe  99.3%   LogReg-hyb  99.6%   MLP-hyb 100.0%
gpt        probe  92.7%   LogReg-hyb  94.2%   MLP-hyb  91.8%
opus       probe  99.4%   LogReg-hyb  99.6%   MLP-hyb 100.0%
qwen       probe  98.3%   LogReg-hyb  98.3%   MLP-hyb  97.7%
sonnet     probe  99.6%   LogReg-hyb  99.8%   MLP-hyb  99.8%

summary (macro-F1):
  probe (reference):  mean 98.1  worst 92.7 (gpt)
  LogReg-hybrid:      mean 98.0  worst 94.2 (gpt)
  fixed-MLP-hybrid:   mean 97.5  worst 91.8 (gpt)

FUSION HYPOTHESIS SUPPORTED: converged hybrid beats the probe on the worst fold

saved ph2_nb6b_ladder.parquet, ph2_nb6b_logo.parquet, ph2_nb6b_best_head.pt


## Notes — how to read each outcome

- **Step 2 ≈ step 1, and LogReg-hybrid LOGO worst > 92.7:** the fusion idea works; the first run's
  deficit was entirely the under-trained head. Track A's headline numbers become this notebook's
  step-4 / MLP-LOGO results (or the LogReg-hybrid ones, whichever the thesis argues from), and Track
  B proceeds with the selected head config.
- **Step 2 ≈ step 1, but neither hybrid arm lifts the worst fold:** features are harmless but not
  helpful on LOGO. Honest neutral — the stress test becomes the decisive robustness evidence, as
  pre-registered.
- **Step 2 < step 1:** the five features actively cost a converged linear model. Then the scaling
  interface (z-scores vs embedding scale) is the suspect: try per-block normalization (standardize
  the 768 block too) before concluding.
- **Step 3 < step 1 even after the fix:** the MLP family itself underfits here; consider arguing the
  thesis from the linear head (simpler, converged, and honest) with the MLP as an ablation.
- The dropout/layernorm sweep table doubles as the "why this head" defence for the viva.